In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import cmocean.cm as cmo
from matplotlib import rc
from matplotlib.offsetbox import AnchoredText

from auxdata import get_multibeam_map_W1, get_multibeam_map_W3
from mapgridder import grid_map
from plot import fancy_2d_hist, binned_statistic_line_plot

rc('font', size=9)
plt.rcParams['font.sans-serif'] = ['Arial'] + plt.rcParams['font.sans-serif'] # Arial as first choice

fig_width = 5.5 
fig_height = 2.5

In [ ]:
def subgrid(fig, gs_ij):
    sgs = gs_ij.subgridspec(2,3, width_ratios=[1, 0.18, 0.05], height_ratios=[0.2,1])  
    axes = [fig.add_subplot(sgs[1,0]),
            fig.add_subplot(sgs[0,0]),
            fig.add_subplot(sgs[1,1]),
            fig.add_subplot(sgs[1,2])]
    return axes

def histogram_with_stats(data, axes, labels, **kwargs):
    fancy_2d_hist(data.water_column, data.error, None, range_grid, error_grid, 
              xlabel='Water column depth (m)', ylabel='Difference (m)', cmap=cmo.matter,
              axes = axes, hspace=0.06, verbose = True, **kwargs);

    # Make grid related to cell size
    axes[0].set_yticks(error_ticks)
    axes[2].set_yticks(error_ticks)
    
    # Add medians in 2D histogram
    centers= np.arange(150,1501,25)
    binned_statistic_line_plot(data.water_column, data.error, centers, line='median', 
                               ax = axes[0], shade=None, min_nbr_of_points=100, step = True, 
                               c='k', linewidth=0.5, linestyle='-', alpha = 0.7)
    
    # Add percentiles in 2D histogram
    def percentile25(values):
        return np.percentile(values, 25)
        
    def percentile75(values):
        return np.percentile(values, 75)

    binned_statistic_line_plot(data.water_column, data.error, centers, line=percentile25, 
                               ax=axes[0], shade=None, min_nbr_of_points=100, step = True, 
                               c='k', linewidth=0.5, linestyle='--', alpha = 0.7)
    binned_statistic_line_plot(data.water_column, data.error, centers, line=percentile75, 
                               ax=axes[0], shade=None, min_nbr_of_points=100, step = True, 
                               c='k', linewidth=0.5, linestyle='--', alpha = 0.7)
    
    # Mark median in 1D histogram
    perc = np.percentile(data.error, 50)
    xlim = axes[2].get_xlim()
    axes[2].plot(np.array(xlim)*10, [perc, perc], 'k', linewidth=0.5)
            
    # Mark percentile 1D histogram
    for q in [25,75]:
        perc = np.percentile(data.error, q)
        axes[2].plot(np.array(xlim)*10, [perc, perc], 'k--', linewidth=0.5)
        axes[2].set_xlim(xlim)
    
    # Annotate
    for (ax, label) in zip(axes, labels):
        ax.add_artist(AnchoredText(label, loc='upper left', prop=dict(weight='bold'), frameon=False, pad=0))

    return        

## Load data

In [ ]:
W1 = get_multibeam_map_W1()
W3 = get_multibeam_map_W3()

mission02 = pd.read_csv('data/derived/NBP2202_02_ice_draft.csv')
mission03 = pd.read_csv('data/derived/NBP2202_03_ice_draft.csv')
mission02_no_interp = pd.read_csv('data/derived/NBP2202_02_no_interp_ice_draft.csv')
mission03_no_interp = pd.read_csv('data/derived/NBP2202_03_no_interp_ice_draft.csv')

## Under ice

In [ ]:
# Ground truth
mb_grid_size = 10 # m
mb_gridded_map = grid_map(pd.concat([W1,W3], axis=0), mb_grid_size)

lons = xr.DataArray(mission03.longitude.values, dims='z')
lats = xr.DataArray(mission03.latitude.values, dims='z')
ground_truth = mb_gridded_map.interp(Lat=lats, Lon=lons)

AUV_lons = xr.DataArray(mission03.AUV_lon.values, dims='z')
AUV_lats = xr.DataArray(mission03.AUV_lat.values, dims='z')
water_column = mission03.AUV_depth - mb_gridded_map.interp(Lat=AUV_lats, Lon=AUV_lons)

ice_shelf = pd.DataFrame({'error': mission03.ice_draft - ground_truth, 
                          'water_column': water_column}
                        ).dropna()

ice_shelf_no_interp = pd.DataFrame({'error': mission03_no_interp.ice_draft - ground_truth, 
                          'water_column': water_column}
                        ).dropna()

print(f'Number of points overlapping with multibeam data: {len(ice_shelf)}')

## Water surface

In [ ]:
lat_min = -74.15 # Only use datapoints north of this to ensure open water

data = pd.concat([mission02, mission03], axis=0)
data_open_water = data[data.latitude>lat_min]
open_water = pd.DataFrame({'error': data_open_water.ice_draft, # Should be 0
                           'water_column': data_open_water.AUV_depth}
                         ).dropna()


data_no_interp = pd.concat([mission02_no_interp, mission03_no_interp], axis=0)
data_open_water_no_interp = data_no_interp[data.latitude>lat_min]
open_water_no_interp = pd.DataFrame({'error': data_open_water_no_interp.ice_draft, # Should be 0
                           'water_column': data_open_water.AUV_depth}
                         ).dropna()

print(f'Number of points in open water: {len(open_water)}')

## Make figure

In [ ]:
fig, ax_dummy = plt.subplots(visible=False, figsize=(1,0.001))

figsize = (fig_width, 0.6*fig_width)
error_grid = np.linspace(-35,35,36)
range_grid = np.linspace(210, 850, 100)

error_ticks = np.arange(-32, +33, 16) # Half of vertical cell size

In [ ]:
rc('font', size=6)

vmax_ice = 42 
vmax_ow  = 102
vmin = 1


def manuscript_subgrid(fig, gs_ij, hspace=0.1, wspace=0.05):
    sgs = gs_ij.subgridspec(3,2, width_ratios=[1, 0.18], height_ratios=[1,1, 0.3], hspace=hspace, wspace=wspace)  
    axes = [fig.add_subplot(sgs[0,0]),
            fig.add_subplot(sgs[0,1]),
            fig.add_subplot(sgs[1,0]),
            fig.add_subplot(sgs[1,1]),
            fig.add_subplot(sgs[2,0]),
           ]
    return axes

def axes_interp(axes):
    return [axes[0], # 2d hist,
            axes[4], # 1d hist x
            axes[1], # 1d hist y
            ax_dummy] # cbar

def axes_no_interp(axes):
    return [axes[2], # 2d hist,
            ax_dummy, # 1d hist x
            axes[3], # 1d hist y
            ax_dummy] # cbar

fig = plt.figure(figsize = (fig_width, fig_width*0.5))

gs = gridspec.GridSpec(1, 2, figure=fig, wspace=0.05)
axes_ice = manuscript_subgrid(fig, gs[0])
axes_ow  = manuscript_subgrid(fig, gs[1])



# 2D histogram 
print('Ice shelf:')
histogram_with_stats(ice_shelf, axes_interp(axes_ice), ['(a)', '(e)', '(b)'], vmin = vmin, vmax = vmax_ice)

print('Ice shelf no interp:')
histogram_with_stats(ice_shelf_no_interp, axes_no_interp(axes_ice), ['(c)', '', '(d)'], vmin = vmin, vmax = vmax_ice)

print('Open ocean:')
histogram_with_stats(open_water, axes_interp(axes_ow), ['(f)', '(j)', '(g)'], vmin = vmin, vmax = vmax_ow)

print('Open ocean no interp:')
histogram_with_stats(open_water_no_interp, axes_no_interp(axes_ow), ['(h)', '', '(i)'], vmin = vmin, vmax = vmax_ow)

# Remove ticks and labels
for axes in [axes_ice, axes_ow]:
    axes[4].set_xlabel('Water column thickness (m)')
    axes[4].set_xticklabels(axes[0].get_xticklabels())
for ax in [axes_ice[i] for i in [1,2]]:
    ax.set_xticklabels([])
    ax.set_xlabel('')
for ax in [axes_ow[i] for i in [1,2]]:
    ax.set_xticklabels([])   
    ax.set_xlabel('')
for ax in [axes_ow[i] for i in [0,2,4]]:
    ax.set_yticklabels([])
    ax.set_ylabel('')
    
# Same limits for 1d histograms
axes_ice[4].set_ylim(axes_ow[4].get_ylim())
axes_ice[3].set_xlim(axes_ice[1].get_xlim())
axes_ow[3].set_xlim(axes_ow[1].get_xlim())

# Adjust xticks open water
for ax in [axes_ow[i] for i in [1,3]]:
    ax.set_xticks([0,1400])

# Change length of all ticks
for ax in axes_ice + axes_ow:
    ax.tick_params('both', length=0)

# colorbars
for (axes, vmax) in zip([axes_ice, axes_ow], [vmax_ice, vmax_ow]):
    im = ax_dummy.scatter(x=1,y=1,c=1,cmap=cmo.matter, vmin=vmin, vmax=vmax)
    cbar = plt.colorbar(im, ax=axes, orientation='horizontal', location='top', pad = 0.02, label = 'Count')
    cbar.ax.tick_params(length=2)

# annotations
axes   = [axes_ice[0], axes_ice[2], axes_ow[0], axes_ow[2]]
labels = ['Ice shelf', 'without interpolation', 'Open water', 'without interpolation']
for (ax, label) in zip(axes, labels):
    ax.add_artist(AnchoredText(label, loc='upper center', prop=dict(weight='bold'), frameon=False, pad=0))

plt.savefig('figures/fig7.png', bbox_inches = 'tight', dpi=600)

### Summarize statistics

In [ ]:
data_list = [ice_shelf, open_water, ice_shelf_no_interp, open_water_no_interp]
label_list = ['ice shelf', 'open water', 'ice shelf no interpolation', 'open water no interpolation']

for (data, label) in zip(data_list, label_list):
    print(f'\n------------ {label} ---------------')
    print(f'Mean error: {np.mean(data.error):.1f}')
    print(f'Median error: {np.median(data.error):.1f}')
    print(f'Standard deviation: {np.std(data.error):.1f}')
    for q in [2.5,25,75,97.5]:
        print(f'{q}th percentile: {np.percentile(data.error, q):.1f}')
    print(f'interquartile range: {np.percentile(data.error, 75)-np.percentile(data.error, 25):.1f}')